### **Alineamiento contrastivo audio-texto y video-texto**

#### **Recuperación cruzada, pooling temporal, temperatura y negativos difíciles**

Este cuaderno construye una línea base controlada para estudiar alineamiento entre texto y secuencias de audio o video. El objetivo es medir cuándo una representación global pierde eventos breves y cuándo una representación condicionada por la consulta mejora la recuperación.

La pregunta central es:

> ¿Qué protocolo permite demostrar que un sistema recupera la evidencia temporal correcta y no solamente una categoría semántica general?


### **Preguntas de investigación**

#### **Hipótesis de trabajo**

**H1.** El promedio temporal diluye eventos breves cuando el fondo ocupa la mayor parte de la secuencia.

**H2.** La atención condicionada por texto mejora la recuperación exacta cuando el evento es localizado.

**H3.** Los negativos de la misma clase son más difíciles que los negativos aleatorios.

**H4.** Una temperatura demasiado alta o demasiado baja degrada la pérdida contrastiva.

**H5.** La mejora promedio puede ocultar fallos sistemáticos en eventos de corta duración.


### **Configuración reproducible**

#### **Importaciones, rutas y semilla**


In [ ]:
from __future__ import annotations

import json
import math
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def set_seed(seed: int) -> int:
    """Fija las semillas utilizadas en el cuaderno."""
    random.seed(seed)
    np.random.seed(seed)
    print("Semilla fijada:", seed)
    return seed

SEED = set_seed(225)
RESULTS_DIR = Path("resultados/cuaderno25_mcc225")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Directorio de resultados:", RESULTS_DIR)


### **Metadatos del experimento**

#### **Registro del protocolo**


In [ ]:
@dataclass
class ExperimentMetadata:
    course: str
    week: str
    notebook: str
    topic: str
    seed: int
    execution_mode: str


metadata = ExperimentMetadata(
    course="MCC225",
    week="Semana 12",
    notebook="Cuaderno25-MCC225",
    topic="Alineamiento contrastivo audio-texto y video-texto",
    seed=SEED,
    execution_mode="CPU con embeddings sintéticos",
)

with open(RESULTS_DIR / "metadatos.json", "w", encoding="utf-8") as file:
    json.dump(asdict(metadata), file, indent=2, ensure_ascii=False)

asdict(metadata)


### **Marco formal**

#### **Pérdida contrastiva bidireccional**

Sean $t_i$ y $m_j$ embeddings normalizados de texto y medio. La similitud es:

$$
s_{ij} = t_i^T m_j.
$$

La pérdida texto a medio es:

$$
\mathcal{L}_{t \rightarrow m}
=
-\frac{1}{B}\sum_i
\log
\frac{\exp(s_{ii}/\tau)}
{\sum_j \exp(s_{ij}/\tau)}.
$$

La versión bidireccional promedia texto a medio y medio a texto. La temperatura $\tau$ controla la concentración de la distribución de similitudes.


### **Conjunto multimodal controlado**

#### **Eventos localizados y fondo dominante**

Cada ejemplo contiene un vector latente individual. El texto representa ese vector. El audio y el video contienen el mismo vector solamente durante un intervalo temporal. El resto de la secuencia contiene fondo y ruido.

Este diseño permite conocer:

1. El candidato correcto.
2. La clase semántica.
3. El intervalo del evento.
4. La duración del evento.
5. Los negativos de la misma clase.


In [ ]:
@dataclass
class MultimodalSample:
    sample_id: str
    class_name: str
    text_embedding: np.ndarray
    audio_sequence: np.ndarray
    video_sequence: np.ndarray
    event_start: int
    event_end: int


def normalize_vector(vector: np.ndarray) -> np.ndarray:
    """Normaliza un vector con la norma euclidiana."""
    norm = float(np.linalg.norm(vector))
    if norm == 0.0:
        return vector.copy()
    return vector / norm


def normalize_rows(matrix: np.ndarray) -> np.ndarray:
    """Normaliza las filas de una matriz."""
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.clip(norms, 1e-12, None)


def generate_multimodal_dataset(
    num_classes: int,
    samples_per_class: int,
    sequence_length: int,
    embedding_dimension: int,
    event_length: int,
    noise_level: float,
    seed: int,
) -> list[MultimodalSample]:
    """Genera ejemplos multimodales con eventos temporales conocidos."""
    rng = np.random.default_rng(seed)
    class_prototypes = normalize_rows(
        rng.normal(size=(num_classes, embedding_dimension))
    )
    samples = []

    for class_index in range(num_classes):
        for sample_index in range(samples_per_class):
            instance_component = rng.normal(size=embedding_dimension)
            latent = normalize_vector(
                class_prototypes[class_index]
                + 0.45 * normalize_vector(instance_component)
            )
            text_embedding = normalize_vector(
                latent + rng.normal(0.0, 0.03, size=embedding_dimension)
            )

            event_start = int(rng.integers(1, sequence_length - event_length - 1))
            event_end = event_start + event_length

            audio_sequence = rng.normal(
                0.0,
                noise_level,
                size=(sequence_length, embedding_dimension),
            )
            video_sequence = rng.normal(
                0.0,
                noise_level,
                size=(sequence_length, embedding_dimension),
            )

            audio_sequence[event_start:event_end] += latent
            video_sequence[event_start:event_end] += 0.85 * latent
            video_sequence[event_start:event_end] += rng.normal(
                0.0,
                0.03,
                size=(event_length, embedding_dimension),
            )

            samples.append(MultimodalSample(
                sample_id=f"ejemplo_{class_index}_{sample_index}",
                class_name=f"clase_{class_index}",
                text_embedding=text_embedding,
                audio_sequence=audio_sequence,
                video_sequence=video_sequence,
                event_start=event_start,
                event_end=event_end,
            ))

    return samples


samples = generate_multimodal_dataset(
    num_classes=4,
    samples_per_class=8,
    sequence_length=24,
    embedding_dimension=16,
    event_length=3,
    noise_level=0.16,
    seed=SEED,
)

print("Número de ejemplos:", len(samples))
print("Forma de una secuencia de audio:", samples[0].audio_sequence.shape)


### **Representaciones temporales**

#### **Promedio global y atención condicionada**

El promedio global trata todos los segmentos como igualmente relevantes. La atención condicionada por consulta utiliza similitud entre el texto y cada segmento:

$$
a_t = \operatorname{softmax}(q^T h_t / \tau_a).
$$

El embedding atendido es:

$$
z = \sum_t a_t h_t.
$$


In [ ]:
def mean_pool(sequence: np.ndarray) -> np.ndarray:
    """Calcula un embedding global mediante promedio temporal."""
    return normalize_vector(sequence.mean(axis=0))


def stable_softmax(values: np.ndarray) -> np.ndarray:
    """Calcula softmax de manera numéricamente estable."""
    shifted = values - np.max(values)
    exponentials = np.exp(shifted)
    return exponentials / np.sum(exponentials)


def query_attention_pool(
    query_embedding: np.ndarray,
    sequence: np.ndarray,
    temperature: float,
) -> tuple[np.ndarray, np.ndarray]:
    """Agrega una secuencia utilizando atención condicionada por consulta."""
    normalized_sequence = normalize_rows(sequence)
    scores = normalized_sequence @ normalize_vector(query_embedding)
    weights = stable_softmax(scores / temperature)
    pooled = weights @ sequence
    return normalize_vector(pooled), weights


def compute_similarity_matrix(
    query_embeddings: np.ndarray,
    candidate_embeddings: np.ndarray,
) -> np.ndarray:
    """Calcula una matriz de similitud coseno."""
    return normalize_rows(query_embeddings) @ normalize_rows(candidate_embeddings).T


text_embeddings = np.vstack([sample.text_embedding for sample in samples])
audio_global = np.vstack([mean_pool(sample.audio_sequence) for sample in samples])
video_global = np.vstack([mean_pool(sample.video_sequence) for sample in samples])

audio_attended_rows = []
video_attended_rows = []
attention_diagnostics = []

for sample in samples:
    audio_embedding, audio_weights = query_attention_pool(
        sample.text_embedding,
        sample.audio_sequence,
        temperature=0.12,
    )
    video_embedding, video_weights = query_attention_pool(
        sample.text_embedding,
        sample.video_sequence,
        temperature=0.12,
    )
    audio_attended_rows.append(audio_embedding)
    video_attended_rows.append(video_embedding)
    attention_diagnostics.append({
        "identificador": sample.sample_id,
        "masa_audio_evento": float(
            audio_weights[sample.event_start:sample.event_end].sum()
        ),
        "masa_video_evento": float(
            video_weights[sample.event_start:sample.event_end].sum()
        ),
        "indice_maximo_audio": int(np.argmax(audio_weights)),
        "indice_maximo_video": int(np.argmax(video_weights)),
    })

audio_attended = np.vstack(audio_attended_rows)
video_attended = np.vstack(video_attended_rows)
attention_table = pd.DataFrame(attention_diagnostics)
attention_table.head()


### **Métricas de recuperación**

#### **Exactitud de ranking y calidad de la lista**

Se evalúa recuperación exacta. Para cada consulta, solamente el ejemplo con el mismo identificador es relevante. Además, se registra si el error recupera un ejemplo de la misma clase, lo que indica confusión semántica fina.


In [ ]:
def rank_candidates(similarity_matrix: np.ndarray) -> np.ndarray:
    """Ordena los candidatos de mayor a menor similitud."""
    return np.argsort(-similarity_matrix, axis=1)


def recall_at_k(ranking: np.ndarray, k: int) -> float:
    """Calcula Recall en las primeras k posiciones para relevancia única."""
    hits = [query_index in ranking[query_index, :k] for query_index in range(len(ranking))]
    return float(np.mean(hits))


def mean_reciprocal_rank(ranking: np.ndarray) -> float:
    """Calcula el rango recíproco medio."""
    reciprocal_ranks = []
    for query_index, row in enumerate(ranking):
        position = int(np.where(row == query_index)[0][0]) + 1
        reciprocal_ranks.append(1.0 / position)
    return float(np.mean(reciprocal_ranks))


def ndcg_at_k(ranking: np.ndarray, k: int) -> float:
    """Calcula nDCG con un único candidato relevante por consulta."""
    scores = []
    for query_index, row in enumerate(ranking):
        positions = np.where(row[:k] == query_index)[0]
        if len(positions) == 0:
            scores.append(0.0)
        else:
            position = int(positions[0])
            scores.append(1.0 / math.log2(position + 2.0))
    return float(np.mean(scores))


def evaluate_retrieval(similarity_matrix: np.ndarray) -> dict[str, float]:
    """Calcula métricas de recuperación cruzada."""
    ranking = rank_candidates(similarity_matrix)
    return {
        "recall_1": recall_at_k(ranking, 1),
        "recall_5": recall_at_k(ranking, 5),
        "recall_10": recall_at_k(ranking, 10),
        "mrr": mean_reciprocal_rank(ranking),
        "ndcg_10": ndcg_at_k(ranking, 10),
    }


protocol_embeddings = {
    "audio_global": audio_global,
    "audio_atendido": audio_attended,
    "video_global": video_global,
    "video_atendido": video_attended,
}

metric_rows = []
similarity_matrices = {}
for protocol_name, media_embeddings in protocol_embeddings.items():
    similarity_matrix = compute_similarity_matrix(text_embeddings, media_embeddings)
    similarity_matrices[protocol_name] = similarity_matrix
    metrics = evaluate_retrieval(similarity_matrix)
    metric_rows.append({"protocolo": protocol_name, **metrics})

metric_table = pd.DataFrame(metric_rows)
metric_table


In [ ]:
plt.figure(figsize=(9, 4))
positions = np.arange(len(metric_table))
plt.bar(positions - 0.18, metric_table["recall_1"], width=0.36, label="Recall@1")
plt.bar(positions + 0.18, metric_table["mrr"], width=0.36, label="MRR")
plt.xticks(positions, metric_table["protocolo"], rotation=20)
plt.ylim(0.0, 1.05)
plt.ylabel("Puntaje")
plt.title("Comparación de protocolos de recuperación")
plt.legend()
plt.tight_layout()
plt.show()


### **Pérdida contrastiva y temperatura**

#### **Concentración de la distribución de similitudes**

Se calcula una pérdida bidireccional para diferentes temperaturas. El objetivo no es entrenar un modelo, sino observar cómo la temperatura modifica la penalización asignada a candidatos difíciles.


In [ ]:
def logsumexp(values: np.ndarray, axis: int) -> np.ndarray:
    """Calcula logaritmo de suma de exponenciales de manera estable."""
    maximum = np.max(values, axis=axis, keepdims=True)
    result = maximum + np.log(np.sum(np.exp(values - maximum), axis=axis, keepdims=True))
    return np.squeeze(result, axis=axis)


def symmetric_contrastive_loss(
    first_embeddings: np.ndarray,
    second_embeddings: np.ndarray,
    temperature: float,
) -> float:
    """Calcula una pérdida contrastiva bidireccional."""
    logits = compute_similarity_matrix(first_embeddings, second_embeddings) / temperature
    diagonal = np.diag(logits)
    first_loss = -np.mean(diagonal - logsumexp(logits, axis=1))
    second_loss = -np.mean(diagonal - logsumexp(logits, axis=0))
    return float((first_loss + second_loss) / 2.0)


temperature_rows = []
for temperature in [0.03, 0.05, 0.08, 0.12, 0.20, 0.35, 0.60]:
    temperature_rows.append({
        "temperatura": temperature,
        "perdida_audio_global": symmetric_contrastive_loss(
            text_embeddings,
            audio_global,
            temperature,
        ),
        "perdida_audio_atendido": symmetric_contrastive_loss(
            text_embeddings,
            audio_attended,
            temperature,
        ),
    })

temperature_table = pd.DataFrame(temperature_rows)
temperature_table


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(
    temperature_table["temperatura"],
    temperature_table["perdida_audio_global"],
    marker="o",
    label="Audio global",
)
plt.plot(
    temperature_table["temperatura"],
    temperature_table["perdida_audio_atendido"],
    marker="o",
    label="Audio atendido",
)
plt.xlabel("Temperatura")
plt.ylabel("Pérdida contrastiva")
plt.title("Sensibilidad de la pérdida a la temperatura")
plt.legend()
plt.tight_layout()
plt.show()


### **Negativos difíciles**

#### **Confusión exacta, semántica y temporal**

Un negativo aleatorio suele pertenecer a otra clase. Un negativo semántico pertenece a la misma clase, pero representa otro ejemplo. Un negativo temporal proviene del mismo ejemplo, pero de un intervalo sin evento.


In [ ]:
def temporal_negative_embedding(
    sample: MultimodalSample,
    modality: str,
) -> np.ndarray:
    """Obtiene un embedding desde segmentos fuera del intervalo del evento."""
    sequence = sample.audio_sequence if modality == "audio" else sample.video_sequence
    mask = np.ones(len(sequence), dtype=bool)
    mask[sample.event_start:sample.event_end] = False
    return mean_pool(sequence[mask])


class_names = [sample.class_name for sample in samples]
hard_negative_rows = []
for protocol_name, similarity_matrix in similarity_matrices.items():
    ranking = rank_candidates(similarity_matrix)
    same_class_errors = 0
    other_class_errors = 0

    for query_index, row in enumerate(ranking):
        top_candidate = int(row[0])
        if top_candidate == query_index:
            continue
        if class_names[top_candidate] == class_names[query_index]:
            same_class_errors += 1
        else:
            other_class_errors += 1

    hard_negative_rows.append({
        "protocolo": protocol_name,
        "errores_misma_clase": same_class_errors,
        "errores_otra_clase": other_class_errors,
    })

hard_negative_table = pd.DataFrame(hard_negative_rows)
hard_negative_table


In [ ]:
temporal_negative_rows = []
for modality in ["audio", "video"]:
    for sample in samples:
        negative_embedding = temporal_negative_embedding(sample, modality)
        positive_sequence = (
            sample.audio_sequence if modality == "audio" else sample.video_sequence
        )
        positive_embedding, _ = query_attention_pool(
            sample.text_embedding,
            positive_sequence,
            temperature=0.12,
        )
        positive_score = float(sample.text_embedding @ positive_embedding)
        negative_score = float(sample.text_embedding @ negative_embedding)
        temporal_negative_rows.append({
            "modalidad": modality,
            "identificador": sample.sample_id,
            "similitud_positiva": positive_score,
            "similitud_negativa_temporal": negative_score,
            "margen": positive_score - negative_score,
        })

temporal_negative_table = pd.DataFrame(temporal_negative_rows)
temporal_negative_table.groupby("modalidad")["margen"].agg(["mean", "std", "min"])


### **Duración del evento y robustez**

#### **Evaluación estratificada**

Se repite el experimento para eventos de diferente duración. La comparación permite verificar si una mejora depende de condiciones favorables y si los eventos breves siguen siendo un punto débil.


In [ ]:
duration_rows = []
for event_length in [1, 2, 3, 5, 8]:
    for current_seed in range(10):
        current_samples = generate_multimodal_dataset(
            num_classes=4,
            samples_per_class=5,
            sequence_length=24,
            embedding_dimension=16,
            event_length=event_length,
            noise_level=0.16,
            seed=1000 + current_seed,
        )
        current_text = np.vstack([sample.text_embedding for sample in current_samples])
        current_global = np.vstack([
            mean_pool(sample.audio_sequence) for sample in current_samples
        ])
        current_attended = np.vstack([
            query_attention_pool(
                sample.text_embedding,
                sample.audio_sequence,
                temperature=0.12,
            )[0]
            for sample in current_samples
        ])

        global_metrics = evaluate_retrieval(
            compute_similarity_matrix(current_text, current_global)
        )
        attended_metrics = evaluate_retrieval(
            compute_similarity_matrix(current_text, current_attended)
        )

        duration_rows.append({
            "duracion_evento": event_length,
            "semilla": current_seed,
            "recall_1_global": global_metrics["recall_1"],
            "recall_1_atendido": attended_metrics["recall_1"],
            "ganancia": attended_metrics["recall_1"] - global_metrics["recall_1"],
        })

duration_table = pd.DataFrame(duration_rows)
duration_summary = duration_table.groupby("duracion_evento", as_index=False).agg(
    recall_1_global=("recall_1_global", "mean"),
    recall_1_atendido=("recall_1_atendido", "mean"),
    ganancia_media=("ganancia", "mean"),
)
duration_summary


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(
    duration_summary["duracion_evento"],
    duration_summary["recall_1_global"],
    marker="o",
    label="Promedio global",
)
plt.plot(
    duration_summary["duracion_evento"],
    duration_summary["recall_1_atendido"],
    marker="o",
    label="Atención condicionada",
)
plt.xlabel("Duración del evento en segmentos")
plt.ylabel("Recall@1 medio")
plt.title("Recuperación según duración del evento")
plt.ylim(0.0, 1.05)
plt.legend()
plt.tight_layout()
plt.show()


### **Incertidumbre mediante bootstrap**

#### **Intervalo de confianza para la ganancia de recuperación**


In [ ]:
def bootstrap_mean_interval(
    values: np.ndarray,
    repetitions: int,
    confidence_level: float,
    seed: int,
) -> tuple[float, float]:
    """Calcula un intervalo bootstrap para la media."""
    rng = np.random.default_rng(seed)
    means = []
    for _ in range(repetitions):
        sample = rng.choice(values, size=len(values), replace=True)
        means.append(float(np.mean(sample)))
    alpha = 1.0 - confidence_level
    return (
        float(np.quantile(means, alpha / 2.0)),
        float(np.quantile(means, 1.0 - alpha / 2.0)),
    )


gain_interval = bootstrap_mean_interval(
    duration_table["ganancia"].to_numpy(),
    repetitions=2000,
    confidence_level=0.95,
    seed=SEED,
)
print("Ganancia media de Recall@1:", float(duration_table["ganancia"].mean()))
print("Intervalo bootstrap del 95 por ciento:", gain_interval)


### **Análisis de errores**

#### **Casos donde el candidato correcto no ocupa la primera posición**


In [ ]:
error_rows = []
for protocol_name, similarity_matrix in similarity_matrices.items():
    ranking = rank_candidates(similarity_matrix)
    for query_index, row in enumerate(ranking):
        top_candidate = int(row[0])
        if top_candidate == query_index:
            continue
        true_position = int(np.where(row == query_index)[0][0]) + 1
        error_rows.append({
            "protocolo": protocol_name,
            "consulta": samples[query_index].sample_id,
            "clase_consulta": samples[query_index].class_name,
            "candidato_superior": samples[top_candidate].sample_id,
            "clase_candidato": samples[top_candidate].class_name,
            "misma_clase": samples[top_candidate].class_name == samples[query_index].class_name,
            "posicion_correcta": true_position,
        })

error_table = pd.DataFrame(error_rows)
error_table.head(10)


### **Extensión opcional con modelos abiertos**

#### **Interfaz para CLAP, ImageBind o un encoder de video**

La extensión real debe conservar el mismo protocolo y sustituir solamente la generación de embeddings. Se recomienda registrar modelo, revisión, frecuencia de audio, frames por segundo, duración y hardware.


In [ ]:
RUN_REAL_MODELS = False

if RUN_REAL_MODELS:
    # Esta sección requiere dependencias y pesos externos.
    # Debe implementarse sin modificar las métricas del protocolo base.
    print("Modo de modelos reales activado.")
else:
    print("Modo reproducible sin descargas externas.")


### **Amenazas a la validez**

#### **Límites de la evidencia experimental**

1. Los embeddings sintéticos no reproducen errores acústicos, visuales ni lingüísticos reales.
2. La consulta se deriva del mismo vector latente que el evento, lo que favorece la identificación exacta.
3. La atención condicionada conoce la consulta y puede sobreajustarse a similitudes accidentales.
4. La relevancia única simplifica escenarios donde varios medios pueden ser correctos.
5. Una mejora en recuperación no demuestra razonamiento causal ni grounding preciso.


### **Conclusiones**

#### **Respuesta a la pregunta central**

La evaluación debe distinguir recuperación semántica general de recuperación temporal exacta. El promedio global puede funcionar cuando el evento domina la secuencia, pero pierde información cuando el evento es breve. La atención condicionada mejora la localización, aunque debe evaluarse con negativos difíciles, diferentes duraciones, múltiples semillas y análisis de errores.


#### **Preguntas a desarrollar**

1. ¿Qué representa la temperatura en una pérdida contrastiva?
2. ¿Por qué Recall@1 no es suficiente para evaluar recuperación?
3. ¿Qué diferencia existe entre un negativo semántico y uno temporal?
4. ¿Cuándo la atención condicionada puede producir una evaluación optimista?
5. ¿Por qué es necesario estratificar por duración del evento?
6. ¿Cómo cambiaría el protocolo cuando existen múltiples candidatos relevantes?
7. ¿Qué evidencia demostraría grounding temporal y no solamente similitud global?.


In [ ]:
# Tus respuestas

### **Referencias principales**

#### **Ruta de lectura**

1. Elizalde et al. **CLAP: Learning Audio Concepts From Natural Language Supervision**.
2. Akbari et al. **VATT: Transformers for Multimodal Self-Supervised Learning from Raw Video, Audio and Text**.
3. Xu et al. **VideoCLIP: Contrastive Pre-training for Zero-shot Video-Text Understanding**.
4. Girdhar et al. **ImageBind: One Embedding Space To Bind Them All**.
5. Zhu et al. **LanguageBind: Extending Video-Language Pretraining to N-modality**.


### **Exportación de resultados**

#### **Evidencia reproducible**


In [ ]:
metric_table.to_csv(
    RESULTS_DIR / "metricas_recuperacion.csv",
    index=False,
    encoding="utf-8",
)
attention_table.to_csv(
    RESULTS_DIR / "diagnostico_atencion.csv",
    index=False,
    encoding="utf-8",
)
temperature_table.to_csv(
    RESULTS_DIR / "sensibilidad_temperatura.csv",
    index=False,
    encoding="utf-8",
)
hard_negative_table.to_csv(
    RESULTS_DIR / "errores_negativos_dificiles.csv",
    index=False,
    encoding="utf-8",
)
temporal_negative_table.to_csv(
    RESULTS_DIR / "margenes_negativos_temporales.csv",
    index=False,
    encoding="utf-8",
)
duration_table.to_csv(
    RESULTS_DIR / "resultados_duracion.csv",
    index=False,
    encoding="utf-8",
)
error_table.to_json(
    RESULTS_DIR / "casos_error.jsonl",
    orient="records",
    lines=True,
    force_ascii=False,
)

summary = {
    "ganancia_media_recall_1": float(duration_table["ganancia"].mean()),
    "intervalo_bootstrap_95": list(gain_interval),
}
with open(RESULTS_DIR / "resumen.json", "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2, ensure_ascii=False)

print("Resultados exportados en:", RESULTS_DIR)
